In [18]:
!pip3 install langchain chromadb openai tiktoken pypdf langchain_openai langchain-community sentence-transformers


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: /Library/Frameworks/Python.framework/Versions/3.14/bin/python3.14 -m pip install --upgrade pip


In [19]:
!pip3 install langchain_huggingface


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: /Library/Frameworks/Python.framework/Versions/3.14/bin/python3.14 -m pip install --upgrade pip


In [20]:
from langchain_huggingface import  HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

In [21]:
from langchain_core.documents import Document
# Create LangChain documents for IPL players

doc1 = Document(
        page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
        metadata={"team": "Royal Challengers Bangalore"}
    )
doc2 = Document(
        page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
        metadata={"team": "Mumbai Indians"}
    )
doc3 = Document(
        page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
        metadata={"team": "Chennai Super Kings"}
    )
doc4 = Document(
        page_content="Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
        metadata={"team": "Mumbai Indians"}
    )
doc5 = Document(
        page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
        metadata={"team": "Chennai Super Kings"}
    )


In [22]:
docs=[doc1,doc2,doc3,doc4,doc5]

In [23]:
vector_store=Chroma(
    embedding_function=HuggingFaceEmbeddings(),
    persist_directory='chroma_db',
    collection_name='sample'
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [26]:
# add documents in vectore store
vector_store.add_documents(docs)

['d9272a2f-6c2a-4cb2-9c5e-a3af8cf1d682',
 '2a4acb7d-8390-44c6-9423-205b2d0f33f8',
 '79c660c3-b984-4fd1-a00e-4cba63549a4b',
 '76012aed-35d0-4a8e-92b0-c934afd7b6b6',
 'de7bae5c-69f5-4be2-a551-93df4bc02fa7']

In [27]:
# view documents
vector_store.get(include=['embeddings','documents','metadatas'])

{'ids': ['d9272a2f-6c2a-4cb2-9c5e-a3af8cf1d682',
  '2a4acb7d-8390-44c6-9423-205b2d0f33f8',
  '79c660c3-b984-4fd1-a00e-4cba63549a4b',
  '76012aed-35d0-4a8e-92b0-c934afd7b6b6',
  'de7bae5c-69f5-4be2-a551-93df4bc02fa7'],
 'embeddings': array([[-0.03964756, -0.00801654, -0.01412727, ...,  0.06814157,
         -0.00116095, -0.02425424],
        [-0.01992602,  0.00161878, -0.00959342, ...,  0.02433116,
         -0.00341396, -0.01722026],
        [-0.04256172,  0.01060321, -0.01305763, ...,  0.04738366,
         -0.01748705,  0.00204667],
        [-0.02734393, -0.03300872,  0.01565377, ...,  0.01462707,
          0.01177984,  0.00556422],
        [ 0.02301696, -0.02011724, -0.00123579, ...,  0.02533479,
          0.00280533, -0.05120278]], shape=(5, 768)),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the mo

In [28]:
# search documents
vector_store.similarity_search(query='who among these are bowler?',
k=2)

[Document(metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
 Document(metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.')]

In [29]:
# search documents with score
vector_store.similarity_search_with_score(query='who among these are bowler?',
k=2)

[(Document(metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  1.2790216207504272),
 (Document(metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  1.2890455722808838)]

In [30]:
# meta data filtering
vector_store.similarity_search_with_score(
    query='',
    filter={'team':'Mumbai Indians'}
)

[(Document(metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  1.9515010118484497),
 (Document(metadata={'team': 'Mumbai Indians'}, page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure."),
  1.9653910398483276)]

In [31]:
# update data
updated_doc1=Document(
    page_content="Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a single season. Despite RCB not winning an IPL title under his captaincy, Kohli's passion and fitness set a benchmark for the league. His ability to chase targets and anchor innings has made him one of the most dependable players in T20 cricket.",
    metadata={"team": "Royal Challengers Bangalore"}
)
vector_store.update_document(document_id='d9272a2f-6c2a-4cb2-9c5e-a3af8cf1d682',document=updated_doc1)

In [32]:
# view documents
vector_store.get(include=['embeddings','documents','metadatas'])

{'ids': ['d9272a2f-6c2a-4cb2-9c5e-a3af8cf1d682',
  '2a4acb7d-8390-44c6-9423-205b2d0f33f8',
  '79c660c3-b984-4fd1-a00e-4cba63549a4b',
  '76012aed-35d0-4a8e-92b0-c934afd7b6b6',
  'de7bae5c-69f5-4be2-a551-93df4bc02fa7'],
 'embeddings': array([[-0.04108334, -0.02528016, -0.0214507 , ...,  0.02231625,
         -0.02287272, -0.00913001],
        [-0.01992602,  0.00161878, -0.00959342, ...,  0.02433116,
         -0.00341396, -0.01722026],
        [-0.04256172,  0.01060321, -0.01305763, ...,  0.04738366,
         -0.01748705,  0.00204667],
        [-0.02734393, -0.03300872,  0.01565377, ...,  0.01462707,
          0.01177984,  0.00556422],
        [ 0.02301696, -0.02011724, -0.00123579, ...,  0.02533479,
          0.00280533, -0.05120278]], shape=(5, 768)),
 'documents': ["Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple ce

In [33]:
# delete documents
vector_store.delete(ids=['d9272a2f-6c2a-4cb2-9c5e-a3af8cf1d682'])

In [34]:
# view documents
vector_store.get(include=['embeddings','documents','metadatas'])

{'ids': ['2a4acb7d-8390-44c6-9423-205b2d0f33f8',
  '79c660c3-b984-4fd1-a00e-4cba63549a4b',
  '76012aed-35d0-4a8e-92b0-c934afd7b6b6',
  'de7bae5c-69f5-4be2-a551-93df4bc02fa7'],
 'embeddings': array([[-0.01992602,  0.00161878, -0.00959342, ...,  0.02433116,
         -0.00341396, -0.01722026],
        [-0.04256172,  0.01060321, -0.01305763, ...,  0.04738366,
         -0.01748705,  0.00204667],
        [-0.02734393, -0.03300872,  0.01565377, ...,  0.01462707,
          0.01177984,  0.00556422],
        [ 0.02301696, -0.02011724, -0.00123579, ...,  0.02533479,
          0.00280533, -0.05120278]], shape=(4, 768)),
 'documents': ["Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
  'MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.',
  'Jasprit Bumrah is